In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from after.autoencoder.networks.SimpleNet2D import AutoEncoder2D
from after.dataset import SimpleDataset

DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
MODEL_PATH = "/data/nils/repos/AFTER/autoencoder_runs/multiphonics_silence_lowadv/export_pca.ts"

teacher_hop = 4096
target_hop = 64
sr = 44100

# model = torch.jit.load(MODEL_PATH, map_location=DEVICE).eval()

import gin 
gin.parse_config_file("/data/nils/repos/AFTER/autoencoder_runs/multiphonics_silence_lowadv/config.gin")
model = AutoEncoder2D()
ckpt="/data/nils/repos/AFTER/autoencoder_runs/multiphonics_silence_lowadv/checkpoint825000.pt"
ckpt = torch.load(ckpt, map_location=DEVICE)
model.load_state_dict(ckpt["model_state"], strict=False)
model.to(DEVICE)

In [ ]:

ds = SimpleDataset("/fast-1/nils/federico/audio_44k")

@torch.no_grad()
def dense_teacher_encode(x, model, teacher_hop=4096, target_hop=64):
    """
    x: [C,T] or [1,C,T]

    Returns:
        z     : [N,D], ordered every target_hop samples
        times : [N], corresponding sample positions
    """
    if x.ndim == 2:
        x = x.unsqueeze(0)
    x = x.to(DEVICE)

    zs = []
    ts = []

    # 4096 / 64 = 64 shifted versions
    for offset in range(0, teacher_hop, target_hop):
        if offset >= x.shape[-1]-teacher_hop:
            break
        out = model.encode(x[..., offset:x.shape[-1]-teacher_hop+offset], return_mean=True)[2]

        # handle encode() -> tensor or (tensor, ...)
        z = out[0] if isinstance(out, (tuple, list)) else out

        # expected [B,D,Tz]
        z = z[0].transpose(0, 1)  # [Tz,D]

        # native teacher positions for this phase
        t = offset + torch.arange(
            z.shape[0], device=z.device
        ) * teacher_hop

        # discard positions beyond original waveform
        valid = t < x.shape[-1]
        zs.append(z[valid])
        ts.append(t[valid])

    # concatenate all phases, then sort chronologically
    z = torch.cat(zs, dim=0)
    times = torch.cat(ts, dim=0)

    order = torch.argsort(times)
    return z[order].cpu(), times[order].cpu()


# ------------------------------------------------------------------
# Generate dense latents for the whole dataset
# ------------------------------------------------------------------

dense_latents = []

for i in range(10):
    x = ds[i]["waveform"]
    x = torch.from_numpy(x).float()

    z, latent_samples = dense_teacher_encode(
        x,
        model,
        teacher_hop=teacher_hop,
        target_hop=target_hop,
    )

    dense_latents.append({
        "latent": z,                 # [Tlatent, D]
        "sample_positions": latent_samples,
    })



In [ ]:

# ------------------------------------------------------------------
# Plot one example: waveform + latent velocity
# ------------------------------------------------------------------

i = 2
x = ds[i]["waveform"]
if x.ndim > 1:
    audio = x.mean(0)  # mono just for visualization
else:
    audio = x

z = dense_latents[i]["latent"]
latent_samples = dense_latents[i]["sample_positions"]

# RMS displacement in latent space between consecutive 64-sample steps
velocity = torch.sqrt(torch.mean((z[1:] - z[:-1]) ** 2, dim=-1))

# velocity corresponds to arrival at the second latent
velocity_samples = latent_samples[1:]

audio_t = np.arange(audio.shape[-1]) / sr
latent_t = velocity_samples.numpy() / sr

fig, ax = plt.subplots(2, 1, figsize=(16, 6), sharex=True)

ax[0].plot(audio_t, audio.squeeze())
ax[0].set_ylabel("Waveform")
ax[0].set_title("Audio")

ax[1].plot(latent_t, velocity.numpy())
ax[1].set_ylabel("Latent velocity")
ax[1].set_xlabel("Time (s)")
ax[1].set_title(r"$\sqrt{\mathrm{mean}((z_t-z_{t-1})^2)}$")

plt.tight_layout()
plt.show()

In [ ]:
phase_idx = (latent_samples[1:] % teacher_hop) // target_hop

phase_velocity = torch.stack([
    velocity[phase_idx == p].mean()
    for p in range(teacher_hop // target_hop)
])

plt.figure(figsize=(12, 4))
plt.plot(
    torch.arange(64) * target_hop,
    phase_velocity,
    marker=".",
)

plt.xlabel("Phase offset (samples)")
plt.ylabel("Mean RMS Δz")
plt.title("Mean latent velocity by teacher phase")
plt.show()

: 

In [ ]:
phase

In [ ]:
fig, ax = plt.subplots(
    5,
    1,
    figsize=(18, 14),
    sharex=True,
    gridspec_kw={"height_ratios": [1.3, 1, 1, 1.8, 2]},
)

for ax_ in ax:
    ax_.set_xlim(5,6)
# waveform
ax[0].plot(audio_t, audio, linewidth=0.7)
ax[0].set_ylabel("Amplitude")
ax[0].set_title("Waveform")

# latent RMS velocity
# ax[1].plot(latent_t, velocity_rms.numpy())
# ax[1].set_ylabel("RMS Δz")
# ax[1].set_title("Latent velocity")

# # cosine change
# ax[2].plot(latent_t, cosine_change.numpy())
# ax[2].set_ylabel("1 - cosine")
# ax[2].set_title("Latent directional change")

# evolution of each latent dimension
z_t = latent_samples.numpy() / sr

for d in range(8):
    ax[3].plot(
        z_t,
        z[:, d].numpy()/z[:,d].max(),
        linewidth=1.0,
        label=f"z{d}",
    )

ax[3].set_ylabel("Latent value")
ax[3].set_title("Latent trajectories")
ax[3].legend(
    ncol=8,
    fontsize=8,
    loc="upper right",
)

# latent heatmap
extent = [
    latent_samples[0].item() / sr,
    latent_samples[-1].item() / sr,
    0,
    z.shape[-1],
]

ax[4].imshow(
    z.T.numpy(),
    aspect="auto",
    origin="lower",
    extent=extent,
    interpolation="nearest",
)

ax[4].set_ylabel("Latent dim")
ax[4].set_xlabel("Time (s)")
ax[4].set_title("Dense teacher latent trajectory")

plt.tight_layout()
plt.show()